## Adding progress bars to Learner

In [1]:
!git clone https://github.com/fastai/course-v3.git
!mv course-v3/nbs/dl2/exp . # Move the exp folder to your current directory

Cloning into 'course-v3'...
remote: Enumerating objects: 5909, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 5909 (delta 9), reused 1 (delta 1), pack-reused 5893 (from 1)
Receiving objects: 100% (5909/5909), 263.04 MiB | 29.90 MiB/s, done.
Resolving deltas: 100% (3259/3259), done.


In [2]:
from exp.nb_09b import *
import time
from fastprogress.fastprogress import master_bar, progress_bar
from fastprogress.fastprogress import format_time

One thing has been missing all this time, and as fun as it is to stare at a blank screen waiting for the results, it's nicer to have some tool to track progress.

## Imagenette data

In [3]:
from fastai.data.external import untar_data, URLs
import pathlib

In [4]:
path = untar_data(URLs.IMAGENETTE_160)

<div><progress max="99003388" value="99008512"></progress> 100.01% [99008512/99003388 00:02&lt;00:00]</div>

In [5]:
tfms = [make_rgb, ResizeFixed(128), to_byte_tensor, to_float_tensor]
bs = 64

il = ImageList.from_files(path, tfms=tfms)
sd = SplitData.split_by_func(il, partial(grandparent_splitter, valid_name='val'))
ll = label_by_func(sd, parent_labeler, proc_y=CategoryProcessor())
data = ll.to_databunch(bs, c_in=3, c_out=10, num_workers=4)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


create 4x 32 filter layers:

In [6]:
nfs = [32]*4

We rewrite the `AvgStatsCallback` to add a line with the names of the things measured and keep track of the time per epoch.

basically the exact same as before, except now we're storing our stats in an array, and we're just passing of the array to logger (which is just a print statement at this state)

In [7]:
# export
class AvgStatsCallback(Callback):
    def __init__(self, metrics):
        self.train_stats,self.valid_stats = AvgStats(metrics,True),AvgStats(metrics,False)

    def begin_fit(self):
        met_names = ['loss'] + [m.__name__ for m in self.train_stats.metrics]
        names = ['epoch'] + [f'train_{n}' for n in met_names] + [
            f'valid_{n}' for n in met_names] + ['time']
        self.logger(names)

    def begin_epoch(self):
        self.train_stats.reset()
        self.valid_stats.reset()
        self.start_time = time.time()

    def after_loss(self):
        stats = self.train_stats if self.in_train else self.valid_stats
        with torch.no_grad(): stats.accumulate(self.run)

    def after_epoch(self):
        stats = [str(self.epoch)]
        for o in [self.train_stats, self.valid_stats]:
            stats += [f'{v:.6f}' for v in o.avg_stats]
        stats += [format_time(time.time() - self.start_time)]
        self.logger(stats)

Then we add the progress bars... with a Callback of course!  
`master_bar` handles the count over the epochs while its child `progress_bar` is looping over all the batches.  
We just create one at the beginning or each epoch/validation phase, and update it at the end of each batch.  
By changing the logger of the `Learner` to the `write` function of the master bar, everything is automatically written there.

Note: this requires fastprogress v0.1.21 or later.

In [8]:
# export
class ProgressCallback(Callback):
    _order=-1
    def begin_fit(self):
        self.mbar = master_bar(range(self.epochs))
        self.mbar.on_iter_begin()
        self.run.logger = partial(self.mbar.write, table=True)

    def after_fit(self): self.mbar.on_iter_end()
    def after_batch(self): self.pb.update(self.iter)
    def begin_epoch   (self): self.set_pb()
    def begin_validate(self): self.set_pb()

    def set_pb(self):
        self.pb = progress_bar(self.dl, parent=self.mbar)
        self.mbar.update(self.epoch)

By making the progress bar a callback, you can easily choose if you want to have them shown or not.

can now add `ProgressCallback` to our `cbfs`:

In [12]:
cbfs = [partial(AvgStatsCallback,accuracy),
        CudaCallback,
        ProgressCallback,
        partial(BatchTransformXCallback, norm_imagenette)]

In [13]:
learn = get_learner(nfs, data, 0.4, conv_layer, cb_funcs=cbfs)

In [14]:
learn.fit(2)

epoch,train_loss,train_accuracy,valid_loss,valid_accuracy,time


epoch,train_loss,train_accuracy,valid_loss,valid_accuracy,time
0,1.811877,0.365614,2.078021,0.363057,00:16
1,1.446718,0.510508,1.421861,0.528408,00:14


/content/exp/nb_08.py:172: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  res = torch.ByteTensor(torch.ByteStorage.from_buffer(item.tobytes()))
/content/exp/nb_08.py:172: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  res = torch.ByteTensor(torch.ByteStorage.from_buffer(item.tobytes()))
/content/exp/nb_08.py:172: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access Un